# Final Submission

Лучшая модель: **LightGBM Tuned (Optuna)** — Val RMSLE = 0.3730.

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib
import os

artifacts_dir = os.path.join('..', 'artifacts')
data_dir = os.path.join('..', 'data')
processed_dir = os.path.join(data_dir, 'processed')
submissions_dir = os.path.join('..', 'submissions')
os.makedirs(submissions_dir, exist_ok=True)

# Загружаем модель и тестовые данные
model = joblib.load(os.path.join(artifacts_dir, 'lgbm_tuned.pkl'))
test_fe = pd.read_parquet(os.path.join(processed_dir, 'test_fe.parquet'))
test_raw = pd.read_csv(os.path.join(data_dir, 'test.csv'))

print(f'Test FE: {test_fe.shape}, Test raw: {test_raw.shape}')
print(f'IDs: {test_raw["id"].min()} — {test_raw["id"].max()}')

Test FE: (28512, 32), Test raw: (28512, 5)
IDs: 3000888 — 3029399


In [2]:
DROP_COLS = ['date', 'sales']
FEATURE_COLS = [c for c in test_fe.columns if c not in DROP_COLS]

X_test = test_fe[FEATURE_COLS]
pred_log = model.predict(X_test, num_iteration=model.best_iteration)
pred = np.expm1(pred_log).clip(0)

# Маппинг: test_fe отсортирован по (store_nbr, family, date), test_raw — по id.
# Сопоставляем по (date, store_nbr, family).
test_fe_keys = test_fe[['date', 'store_nbr', 'family']].copy()
test_fe_keys['sales'] = pred

test_raw['date'] = pd.to_datetime(test_raw['date'])

# Восстанавливаем оригинальные family-коды из label encoder
encoders = joblib.load(os.path.join(artifacts_dir, 'label_encoders.pkl'))
le_family = encoders['family']
family_map = {idx: cls for idx, cls in enumerate(le_family.classes_)}
test_fe_keys['family_orig'] = test_fe_keys['family'].map(family_map)

submission = test_raw[['id', 'date', 'store_nbr', 'family']].merge(
    test_fe_keys[['date', 'store_nbr', 'family_orig', 'sales']],
    left_on=['date', 'store_nbr', 'family'],
    right_on=['date', 'store_nbr', 'family_orig'],
    how='left'
)

assert submission['sales'].isna().sum() == 0, f'Missing predictions: {submission["sales"].isna().sum()}'
assert len(submission) == 28512, f'Wrong row count: {len(submission)}'

submission_final = submission[['id', 'sales']].sort_values('id')

out_path = os.path.join(submissions_dir, 'submission.csv')
submission_final.to_csv(out_path, index=False)

print(f'Submission saved: {out_path}')
print(f'Shape: {submission_final.shape}')
print(f'Sales — mean: {submission_final["sales"].mean():.2f}, min: {submission_final["sales"].min():.2f}, max: {submission_final["sales"].max():.2f}')
print(f'Zeros: {(submission_final["sales"] == 0).sum()} ({(submission_final["sales"] == 0).mean():.1%})')
print(submission_final.head(10))

Submission saved: ..\submissions\submission.csv
Shape: (28512, 2)
Sales — mean: 210.65, min: 0.00, max: 14804.61
Zeros: 1811 (6.4%)
        id        sales
0  3000888     4.281424
1  3000889     0.000000
2  3000890     4.987582
3  3000891  2368.245828
4  3000892     0.000000
5  3000893   409.750831
6  3000894    10.528026
7  3000895   780.045739
8  3000896   870.232301
9  3000897   153.476581
